# LOAD AND INSPECT

In [53]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [54]:
df = pd.read_csv('../data/Enhanced_Mandal_Farm.csv')

In [55]:
# print("Shape",df.shape)
# print("Describe",df.describe())
# print("Columns",df.columns.tolist())
# print("Missing values",df.isnull().sum())
# print("Data types",df.dtypes)
print(df.head(3))

         Date    Pond_ID Pond_Type Target_Species  Season  Water_Temp_C  \
0  2025-01-01  GrowOut-A   Farming      Pangasius  Winter          17.0   
1  2025-01-02  GrowOut-A   Farming      Pangasius  Winter          15.7   
2  2025-01-03  GrowOut-A   Farming      Pangasius  Winter          15.5   

  Weather_Condition  Rainfall_mm  pH_Level  Ammonia_ppm  ...  Fish_Count  \
0            Cloudy          0.0       7.9         0.05  ...       10000   
1             Sunny          0.0       6.6         0.02  ...        9999   
2             Sunny          0.0       7.4         0.03  ...        9996   

   Daily_Feed_kg  Est_Avg_Weight_g  Feed_Cost_NPR  Labor_Cost_NPR  \
0            4.4              50.3            442             523   
1            4.2              50.5            443             512   
2            5.0              50.9            513             491   

   Market_Price_NPR  Estimated_Revenue_NPR  Daily_Profit_Loss_NPR  \
0               213                 107139      

# dATA clean

In [56]:
def clean_data(df):
    df= df.copy()


df['Date'] = pd.to_datetime(df['Date'])

df.dtypes

Date                     datetime64[us]
Pond_ID                             str
Pond_Type                           str
Target_Species                      str
Season                              str
Water_Temp_C                    float64
Weather_Condition                   str
Rainfall_mm                     float64
pH_Level                        float64
Ammonia_ppm                     float64
Dissolved_Oxygen_mgL            float64
Mortality_Count                   int64
Fish_Count                        int64
Daily_Feed_kg                   float64
Est_Avg_Weight_g                float64
Feed_Cost_NPR                     int64
Labor_Cost_NPR                    int64
Market_Price_NPR                  int64
Estimated_Revenue_NPR             int64
Daily_Profit_Loss_NPR             int64
Stocking_Density                    str
Harvest_Ready                       str
dtype: object

## fill missing numeric columns with median per pond

In [57]:
num_cols = ['Water_Temp_C', 'pH_Level','Ammonia_ppm','Dissolved_Oxygen_mgL','Daily_Feed_kg','Est_Avg_Weight_g']


for col in num_cols:
    df[col] = df.groupby('Pond_ID')[col].transform(
        lambda x: x.fillna(x.median())
    )



## Cap outliers using IQR per pond

In [58]:
def cap_outliers(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3-Q1
    return series.clip(Q1 - 1.5*IQR, Q3+ 1.5*IQR)


for col in ['Mortality_Count', 'Ammonia_ppm','Dissolved_Oxygen_mgL']:
    df[col] = df.groupby('Pond_ID')[col].transform(cap_outliers)





## fix negative profits to 0 where fish_counts = 0 (pond empty)

In [59]:
df.loc[df['Fish_Count'] == 0, 'Daily_Profit_Loss_NPR'] = 0



In [61]:

print("Cleaned. Sahpe", df.shape)
print("Nulls remaining:", df.isnull().sum().sum())

Cleaned. Sahpe (365, 22)
Nulls remaining: 0
